In [2]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.integrate import odeint
import numpy as np

The system of equations in question model atmospheric convection:
- x is related to the rate of fluid flow
- y is related to the temperature difference
- z is related to the nonlinearity of the "vertical temperature profile"

$$\frac{dx}{dt} = \sigma (y - x)$$

$$\frac{dy}{dt} = x (\rho - z) - y$$

$$\frac{dz}{dt} = x y - \beta z$$

In [ ]:
# this will take large inspiration from the n-body code given to
# my mechanics class by Larry. this will translate well since we'll be
# simulating n particles subject to convection

def ode_system(inputs,t,m):
    """
    This function represents a series of first order ODEs.

    Return: List of expressions for the first time derivative of the inputs, in order.
    """

    # Parse the inputs list to positions x,y and vector magnitudes xdot,ydot
    # But this time, let's store them in lists since there will be N
    # values of each of these for N particles.

    x = []
    y = []
    xdot = []
    ydot = []

    Fx = []
    Fy = []

    # Let's parse the inputs
    # This expects that the inputs are ordered as:
    # [x1,y1,xdot1,ydot1,x2,y2,xdot2,ydot2,...]
    for inputIndex in range(0,len(inputs),4):
        x.append(inputs[inputIndex])
        y.append(inputs[inputIndex+1])
        xdot.append(inputs[inputIndex+2])
        ydot.append(inputs[inputIndex+3])

    # Now the hard part is figuring out the sum of forces for each of these.
    # It's no longer going to be a simple xdot=Fx/m
    # We now need to sum together the gravitational forces of all of the other
    # particles.

    # Let's assume the force on particle i from particle j is
    # F_ij = -(mi*mj)/rij^2

    # So this loop is over each particle i
    for iparticle in range(len(x)):
        # The forces on particle i in the x and y directions.
        # Let's initialize them as zero and then add up the components
        Fix = 0
        Fiy = 0

        # Position of i:
        xi = x[iparticle]
        yi = y[iparticle]

        # We also need the mass of the particles... Let's assume they were
        # stored in a list in the m object.
        mi = m[iparticle]

        for jparticle in range(len(x)):
            # in this double loop, we want to skip when iparticle and jparticle
            # are the same particle. we're saying there is no gravitational self-interaction
            if iparticle==jparticle:
                continue
            # Position of j
            xj = x[jparticle]
            yj = y[jparticle]

            mj = m[jparticle]

            # Now we have all the info to calculate the distance between i and j
            rij = np.sqrt((xi-xj)**2 + (yi-yj)**2)


            # The magnitude of this force will be
            # -(m1*m2)/rij^2
            # But... we need Fx and Fy. So that's then some trig. We did some
            # of this last time:
            phi = np.arctan2(yi-yj,xi-xj)
            Fijr = -mi*mj/(rij*rij)
            Fijx = Fijr*np.cos(phi)
            Fijy = Fijr*np.sin(phi)

            # and add them to our running Fix and Fiy sums.
            # (Google the += operator in python if you haven't seen this before)
            Fix += Fijx
            Fiy += Fijy

        # Now we have all of the forces that act on particle i
        # Let's store them in a list
        Fx.append(Fix)
        Fy.append(Fiy)

    # Let's make a list of values to return
    # this will have to be ordered like:
    # [xdot1, ydot1, F1x/m1, F1y/m1, xdot2, ydot2, F2x/m2, F2y/m2, ...]

    returnlist = []
    for iparticle in range(len(x)):
        returnlist.append(xdot[iparticle])
        returnlist.append(ydot[iparticle])
        returnlist.append(Fx[iparticle]/m[iparticle])
        returnlist.append(Fy[iparticle]/m[iparticle])

    return returnlist



nframes = 50
tmax = 100
t_array = np.linspace(0,tmax,nframes)

mlist = [1,1]

solutions = odeint(ode_system, (-2,0,0,-0.4,2,0,0,0.4), t_array, args=(mlist,))

x1_array = solutions[:,0]
y1_array = solutions[:,1]

x2_array = solutions[:,4]
y2_array = solutions[:,5]


plotSomeStuff([x1_array,x2_array],[y1_array,y2_array],nframes)